<a href="https://colab.research.google.com/github/MajetyCharanSaiSujith/BlundleShrink/blob/main/Modell.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# BUNDLEShrink - GITHUB FEATURE EXTRACTOR
# FIXED VERSION FOR VITE / WEBPACK / ROLLUP PROJECTS
# ============================================================

!apt-get update -qq
!apt-get install -y -qq brotli

import os
import json
import shutil
import subprocess
import gzip
import brotli
import re
import pandas as pd


# ============================================================
# 1. REPOSITORY
# ============================================================

REPO_URL = "https://github.com/AbhinayAmbati/OnlineTicketBookingSystem"

PROJECT_DIR = "/content/OnlineTicketBookingSystem"


# ============================================================
# 2. CLONE
# ============================================================

if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)

print("Cloning repository...")

result = subprocess.run(
    [
        "git",
        "clone",
        "--depth",
        "1",
        REPO_URL,
        PROJECT_DIR
    ],
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print(result.stderr)
    raise Exception("Git clone failed")

print("Repository cloned successfully.\n")


# ============================================================
# 3. FIND PACKAGE.JSON
# ============================================================

package_files = []

for path, dirs, files in os.walk(PROJECT_DIR):

    dirs[:] = [
        d for d in dirs
        if d not in ["node_modules", ".git"]
    ]

    if "package.json" in files:

        package_files.append(
            os.path.join(
                path,
                "package.json"
            )
        )


print("package.json files:")

for p in package_files:
    print(p)


# ============================================================
# 4. FIND FRONTEND
# ============================================================

frontend_package = None

for p in package_files:

    p2 = p.replace("\\", "/")

    if "/frontend/" in p2:

        frontend_package = p

        break


if frontend_package is None:

    root_package = os.path.join(
        PROJECT_DIR,
        "package.json"
    )

    if os.path.exists(root_package):

        frontend_package = root_package


if frontend_package is None:

    raise Exception(
        "Frontend package.json not found"
    )


frontend_dir = os.path.dirname(
    frontend_package
)


print("\nFrontend:")
print(frontend_dir)


# ============================================================
# 5. READ PACKAGE.JSON
# ============================================================

with open(
    frontend_package,
    "r",
    encoding="utf-8"
) as f:

    package = json.load(f)


package_name = package.get(
    "name",
    "Unknown"
)

dependencies = package.get(
    "dependencies",
    {}
)

dev_dependencies = package.get(
    "devDependencies",
    {}
)

scripts = package.get(
    "scripts",
    {}
)


# THIS FIXES YOUR ERROR
dependency_count = len(
    dependencies
)


dev_dependency_count = len(
    dev_dependencies
)


print("\nPackage:", package_name)

print(
    "Dependencies:",
    dependency_count
)

print(
    "Dev dependencies:",
    dev_dependency_count
)


# ============================================================
# 6. BUILD COMMAND
# ============================================================

if "build" not in scripts:

    raise Exception(
        "No npm build script found"
    )


build_command = [
    "npm",
    "run",
    "build"
]


print(
    "\nBuild command:",
    " ".join(build_command)
)


# ============================================================
# 7. INSTALL
# ============================================================

print("\nInstalling dependencies...")

lock_file = os.path.join(
    frontend_dir,
    "package-lock.json"
)


if os.path.exists(lock_file):

    install_command = [
        "npm",
        "ci",
        "--ignore-scripts"
    ]

else:

    install_command = [
        "npm",
        "install",
        "--ignore-scripts"
    ]


result = subprocess.run(
    install_command,
    cwd=frontend_dir,
    capture_output=True,
    text=True
)


if result.returncode != 0:

    print(result.stdout)
    print(result.stderr)

    raise Exception(
        "npm installation failed"
    )


print(
    "Dependencies installed successfully."
)


# ============================================================
# 8. BUILD
# ============================================================

print("\nBuilding frontend...\n")


result = subprocess.run(
    build_command,
    cwd=frontend_dir,
    capture_output=True,
    text=True
)


print(result.stdout)


if result.returncode != 0:

    print(result.stderr)

    raise Exception(
        "Frontend build failed"
    )


print(
    "\nFrontend build successful."
)


# ============================================================
# 9. FIND BUILD DIRECTORY
# ============================================================

possible_dirs = [
    "dist",
    "build",
    "out",
    ".next"
]


build_dir = None


for d in possible_dirs:

    p = os.path.join(
        frontend_dir,
        d
    )

    if os.path.isdir(p):

        build_dir = p

        break


if build_dir is None:

    raise Exception(
        "Build output directory not found"
    )


print(
    "\nBuild directory:"
)

print(
    build_dir
)


# ============================================================
# 10. FIND JAVASCRIPT FILES
# ============================================================

js_files = []


for path, dirs, files in os.walk(
    build_dir
):

    for f in files:

        if f.lower().endswith(
            ".js"
        ):

            js_files.append(
                os.path.join(
                    path,
                    f
                )
            )


print(
    "\nJavaScript files:",
    len(js_files)
)


# ============================================================
# 11. RAW JS SIZE
# ============================================================

total_js_bytes = 0


for f in js_files:

    total_js_bytes += os.path.getsize(
        f
    )


bundle_kb = (
    total_js_bytes / 1024
)


bundle_mb = (
    bundle_kb / 1024
)


# ============================================================
# 12. GZIP + BROTLI
# ============================================================

total_gzip_bytes = 0

total_brotli_bytes = 0


for f in js_files:

    with open(
        f,
        "rb"
    ) as file:

        data = file.read()


    gz = gzip.compress(
        data,
        compresslevel=9
    )


    br = brotli.compress(
        data,
        quality=11
    )


    total_gzip_bytes += len(gz)

    total_brotli_bytes += len(br)


gzip_kb = (
    total_gzip_bytes / 1024
)


brotli_kb = (
    total_brotli_bytes / 1024
)


# ============================================================
# 13. SOURCE MODULE COUNT
# ============================================================

source_extensions = (
    ".js",
    ".jsx",
    ".ts",
    ".tsx"
)


module_count = 0


src_dir = os.path.join(
    frontend_dir,
    "src"
)


if os.path.exists(src_dir):

    for path, dirs, files in os.walk(
        src_dir
    ):

        for f in files:

            if f.endswith(
                source_extensions
            ):

                module_count += 1


# ============================================================
# 14. NPM DEPENDENCY TREE
# ============================================================

print(
    "\nAnalyzing dependency tree..."
)


result = subprocess.run(

    [
        "npm",
        "ls",
        "--all",
        "--json"
    ],

    cwd=frontend_dir,

    capture_output=True,

    text=True
)


versions = {}


def collect_versions(node):

    deps = node.get(
        "dependencies",
        {}
    )

    for name, info in deps.items():

        version = info.get(
            "version"
        )

        if version:

            if name not in versions:

                versions[name] = set()

            versions[name].add(
                version
            )

        collect_versions(info)


try:

    tree = json.loads(
        result.stdout
    )

    collect_versions(tree)

except Exception as e:

    print(
        "Could not parse npm tree:",
        e
    )


duplicate_packages = 0

duplicate_versions = 0


for name, version_set in versions.items():

    if len(version_set) > 1:

        duplicate_packages += 1

        duplicate_versions += (
            len(version_set) - 1
        )


# ============================================================
# 15. SOURCE FILE COUNT
# ============================================================

source_files = []


if os.path.exists(src_dir):

    for path, dirs, files in os.walk(
        src_dir
    ):

        for f in files:

            if f.endswith(
                source_extensions
            ):

                source_files.append(
                    os.path.join(
                        path,
                        f
                    )
                )


module_count = len(
    source_files
)


# ============================================================
# 16. SIMPLE DEAD CODE ESTIMATE
# ============================================================

all_source = ""


for f in source_files:

    try:

        with open(
            f,
            "r",
            encoding="utf-8",
            errors="ignore"
        ) as file:

            all_source += (
                "\n" +
                file.read()
            )

    except:

        pass


unused_files = 0

unused_bytes = 0


for f in source_files:

    basename = os.path.splitext(
        os.path.basename(f)
    )[0]


    if basename in [
        "main",
        "index",
        "App"
    ]:

        continue


    pattern = (
        r"(from\s+['\"].*"
        + re.escape(basename)
        + r"|import\s+.*"
        + re.escape(basename)
        + r")"
    )


    if not re.search(
        pattern,
        all_source
    ):

        unused_files += 1

        try:

            unused_bytes += os.path.getsize(
                f
            )

        except:

            pass


dead_code_kb = (
    unused_bytes / 1024
)


# ============================================================
# 17. TREE-SHAKING ESTIMATE
# ============================================================

if bundle_kb > 0:

    tree_shaking = max(
        0,
        min(
            100,
            (
                1 -
                dead_code_kb /
                bundle_kb
            ) * 100
        )
    )

else:

    tree_shaking = 0


# ============================================================
# 18. CREATE FEATURE DATA
# ============================================================

features = {

    "project": package_name,

    "repository": REPO_URL,

    "bundle_before": round(
        bundle_kb,
        2
    ),

    "gzip_before": round(
        gzip_kb,
        2
    ),

    "brotli_before": round(
        brotli_kb,
        2
    ),

    "dependencies": dependency_count,

    "duplicates": duplicate_versions,

    "duplicate_packages": duplicate_packages,

    "dead_code": round(
        dead_code_kb,
        2
    ),

    "tree_shaking": round(
        tree_shaking,
        2
    ),

    "modules": module_count,

    "js_files": len(
        js_files
    )
}


# ============================================================
# 19. DATAFRAME
# ============================================================

df = pd.DataFrame(
    [features]
)


# ============================================================
# 20. DISPLAY
# ============================================================

print(
    "\n"
    "================================================"
)

print(
    "           BUNDLEShrink ANALYSIS"
)

print(
    "================================================"
)

display(df)


# ============================================================
# 21. SAVE
# ============================================================

output = (
    "/content/"
    "bundleshrink_features.csv"
)


df.to_csv(
    output,
    index=False
)


# ============================================================
# 22. SUMMARY
# ============================================================

print(
    "\n================================================"
)

print(
    "                 SUMMARY"
)

print(
    "================================================"
)

print(
    f"Project              : {package_name}"
)

print(
    f"JS Bundle            : {bundle_kb:.2f} KB"
)

print(
    f"JS Bundle            : {bundle_mb:.2f} MB"
)

print(
    f"Gzip                 : {gzip_kb:.2f} KB"
)

print(
    f"Brotli               : {brotli_kb:.2f} KB"
)

print(
    f"Dependencies         : {dependency_count}"
)

print(
    f"Duplicate versions   : {duplicate_versions}"
)

print(
    f"Duplicate packages   : {duplicate_packages}"
)

print(
    f"Modules              : {module_count}"
)

print(
    f"JS files             : {len(js_files)}"
)

print(
    f"Dead-code estimate   : {dead_code_kb:.2f} KB"
)

print(
    f"Tree-shaking estimate: {tree_shaking:.2f}%"
)

print(
    "================================================"
)


# ============================================================
# 23. DOWNLOAD
# ============================================================

from google.colab import files

files.download(
    output
)

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package brotli.
(Reading database ... 122809 files and directories currently installed.)
Preparing to unpack .../brotli_1.1.0-2build2_amd64.deb ...
Unpacking brotli (1.1.0-2build2) ...
Setting up brotli (1.1.0-2build2) ...
Processing triggers for man-db (2.12.0-4build2) ...
Cloning repository...
Repository cloned successfully.

package.json files:
/content/OnlineTicketBookingSystem/frontend/package.json

Frontend:
/content/OnlineTicketBookingSystem/frontend

Package: frontend
Dependencies: 6
Dev dependencies: 11

Build command: npm run build

Installing dependencies...
Dependencies installed successfully.

Building frontend...


> frontend@0.0.0 build
> vite build

vite v5.4.20 building for production...
transforming...
✓ 796 modules transformed.
rendering chunks...
co

,project,repository,bundle_before,gzip_before,brotli_before,dependencies,duplicates,duplicate_packages,dead_code,tree_shaking,modules,js_files
0,frontend,https://github.com/AbhinayAmbati/OnlineTicketB...,491.84,117.46,97.43,6,11,11,5.07,98.97,48,1



                 SUMMARY
Project              : frontend
JS Bundle            : 491.84 KB
JS Bundle            : 0.48 MB
Gzip                 : 117.46 KB
Brotli               : 97.43 KB
Dependencies         : 6
Duplicate versions   : 11
Duplicate packages   : 11
Modules              : 48
JS files             : 1
Dead-code estimate   : 5.07 KB
Tree-shaking estimate: 98.97%


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>